# Transfert d'apprentissage — EfficientNetB0

## Chargement du split (calculé dans eda_split.ipynb)

In [1]:
import pandas as pd

df_split = pd.read_csv(r"C:\Users\DELL\PneumoScan\data\split.csv")

train_paths = df_split[df_split["split"] == "train"]["path"].values
val_paths = df_split[df_split["split"] == "val"]["path"].values
test_paths = df_split[df_split["split"] == "test"]["path"].values

train_labels = df_split[df_split["split"] == "train"]["label"].values
val_labels = df_split[df_split["split"] == "val"]["label"].values
test_labels = df_split[df_split["split"] == "test"]["label"].values

y_train = (train_labels == "PNEUMONIA").astype(int)
y_val = (val_labels == "PNEUMONIA").astype(int)
y_test = (test_labels == "PNEUMONIA").astype(int)

print(len(train_paths), len(val_paths), len(test_paths))

4252 724 880


## Conversion des images 


In [2]:
import numpy as np
from PIL import Image
from tensorflow.keras.applications.efficientnet import preprocess_input

def preprocess_images_effnet(paths, size=(224, 224)):
    images = []
    for p in paths:
        img = Image.open(p).convert("RGB")
        img = img.resize(size)
        arr = np.array(img, dtype=np.float32)
        images.append(arr)
    images = np.array(images)
    images = preprocess_input(images)
    return images

X_train_eff = preprocess_images_effnet(train_paths)
X_val_eff = preprocess_images_effnet(val_paths)
X_test_eff = preprocess_images_effnet(test_paths)

print(X_train_eff.shape, X_val_eff.shape, X_test_eff.shape)


(4252, 224, 224, 3) (724, 224, 224, 3) (880, 224, 224, 3)


## Construction du modèle avec EfficientNt


In [3]:
%%time
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(
    include_top = False , 
    weights = 'imagenet' , 
    input_shape=(224 , 224 , 3)
)

base_model.trainable = False

model_eff = models.Sequential([
    base_model , 
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1 , activation = 'sigmoid')
])
model_eff.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_eff.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)          │ (None, 7, 7, 1280)          │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,213,668 (16.07 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

CPU times: total: 4.14 s
Wall time: 4.19 s


## Entrainement du modèle


In [4]:
%%time
import numpy as np 
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping 

early_stop = EarlyStopping(
    monitor = 'val_loss' ,
    patience = 3 , 
    restore_best_weights = True
)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

print(class_weight_dict)

history_eff = model_eff.fit(
    X_train_eff, y_train,
    validation_data=(X_val_eff, y_val),
    epochs=10,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stop]
)


{0: np.float64(1.9016100178890876), 1: np.float64(0.6783663050414805)}
Epoch 1/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 130s 789ms/step - accuracy: 0.8801 - loss: 0.2810 - val_accuracy: 0.9309 - val_loss: 0.1711
Epoch 2/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 93s 697ms/step - accuracy: 0.9207 - loss: 0.1928 - val_accuracy: 0.9254 - val_loss: 0.1741
Epoch 3/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 95s 716ms/step - accuracy: 0.9341 - loss: 0.1734 - val_accuracy: 0.9558 - val_loss: 0.1239
Epoch 4/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 140s 698ms/step - accuracy: 0.9370 - loss: 0.1574 - val_accuracy: 0.9530 - val_loss: 0.1225
Epoch 5/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 97s 730ms/step - accuracy: 0.9440 - loss: 0.1377 - val_accuracy: 0.9323 - val_loss: 0.1676
Epoch 6/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 99s 741ms/step - accuracy: 0.9452 - loss: 0.1408 - val_accuracy: 0.9434 - val_loss: 0.1381
Epoch 7/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 104s 779ms/step - accuracy: 0.9523 - loss: 0.1355 - val_accuracy: 0.9337 - val_loss: 0.1672
CPU times

## Métriques

In [5]:
%%time
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, recall_score, confusion_matrix

y_proba_eff = model_eff.predict(X_val_eff).ravel()
y_pred_eff = (y_proba_eff >= 0.25).astype(int)

print("Accuracy :", accuracy_score(y_val, y_pred_eff))
print("Precision :", precision_score(y_val, y_pred_eff))
print("Recall :", recall_score(y_val, y_pred_eff))
print("ROC-AUC :", roc_auc_score(y_val, y_proba_eff))
print("Matrice de confusion :\n", confusion_matrix(y_val, y_pred_eff))

23/23 ━━━━━━━━━━━━━━━━━━━━ 25s 857ms/step
Accuracy : 0.9488950276243094
Precision : 0.9514925373134329
Recall : 0.9788867562380038
ROC-AUC : 0.9901288730463395
Matrice de confusion :
 [[177  26]
 [ 11 510]]
CPU times: total: 1min 31s
Wall time: 25.6 s


## Évaluation finale sur le test set (seuil figé à 0.25, décidé sur val)

from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, recall_score, confusion_matrix

y_proba_test = model_eff.predict(X_test_eff).ravel()
y_pred_test = (y_proba_test >= 0.25).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred_test))
print("Precision :", precision_score(y_test, y_pred_test))
print("Recall :", recall_score(y_test, y_pred_test))
print("ROC-AUC :", roc_auc_score(y_test, y_proba_test))
print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred_test))

### Sauvegarde de modèle entrainé

In [7]:
model_eff.save(r"C:\Users\DELL\PneumoScan\ai-service\models\efficientnetb0_pneumonia.keras")
